# 01 — Inventory and splits (Day 1)

Thin runner: no logic here, only orchestration. All logic lives in
`sharp_har/` (inventory.py, windowing.py, splits.py).
Ref. `giorno1_inventory_splits_SPEC.md`.

## 1. Environment setup

In [1]:
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"
REPO_DIR = Path("/content/sharp-har")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

!pip install -q -r {REPO_DIR / "requirements.txt"}

sys.path.insert(0, str(REPO_DIR))

from sharp_har.utils import set_seed, get_git_hash, read_yaml

set_seed(42)
print("git hash:", get_git_hash(REPO_DIR))

git hash: ed427173cd287d1649743ee1e0c3905c9a989058


## 2. Mount Drive + staging

Data never enters the repo: it lives on Drive, gets copied and
unpacked locally. Training only reads from `/content`.

In [2]:
import time
import zipfile
from google.colab import drive

drive.mount("/content/drive")

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
stage_dir.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for zip_name in paths_cfg["zips"]:
    src = drive_root / zip_name
    dst = Path("/content") / zip_name
    print(f"copying {src} -> {dst}")
    subprocess.run(["cp", str(src), str(dst)], check=True)
    with zipfile.ZipFile(dst) as zf:
        zf.extractall(stage_dir)
staging_seconds = time.time() - t0
print(f"staging completed in {staging_seconds:.1f}s (a datapoint for the day-2 gate)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip -> /content/doppler_traces.zip
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip -> /content/doppler_traces_S4_S5.zip
staging completed in 42.4s (a datapoint for the day-2 gate)


## 3. Naming reconnaissance

**Inspect the output before proceeding.** If the patterns don't
match `FILENAME_PATTERN` in `sharp_har/inventory.py`, fix the regex
there before running the next cell.

In [3]:
from sharp_har.inventory import list_files, print_naming_patterns

files = list_files(stage_dir)
print(f"{len(files)} files found in {stage_dir}")
print_naming_patterns(files, n_examples=30)

408 files found in /content/data
S1a_C_stream_0.txt
S1a_C_stream_1.txt
S1a_C_stream_2.txt
S1a_C_stream_3.txt
S1a_E_stream_0.txt
S1a_E_stream_1.txt
S1a_E_stream_2.txt
S1a_E_stream_3.txt
S1a_H_stream_0.txt
S1a_H_stream_1.txt
S1a_H_stream_2.txt
S1a_H_stream_3.txt
S1a_J1_stream_0.txt
S1a_J1_stream_1.txt
S1a_J1_stream_2.txt
S1a_J1_stream_3.txt
S1a_J2_stream_0.txt
S1a_J2_stream_1.txt
S1a_J2_stream_2.txt
S1a_J2_stream_3.txt
S1a_L_stream_0.txt
S1a_L_stream_1.txt
S1a_L_stream_2.txt
S1a_L_stream_3.txt
S1a_R_stream_0.txt
S1a_R_stream_1.txt
S1a_R_stream_2.txt
S1a_R_stream_3.txt
S1a_S_stream_0.txt
S1a_S_stream_1.txt

31 distinct patterns (digits replaced by '#'):
  S#a_C#_stream_#.txt
  S#a_C_stream_#.txt
  S#a_E_stream_#.txt
  S#a_H#_stream_#.txt
  S#a_H_stream_#.txt
  S#a_J#_stream_#.txt
  S#a_J_stream_#.txt
  S#a_L#_stream_#.txt
  S#a_LOS_stream_#.txt
  S#a_L_stream_#.txt
  S#a_R#_stream_#.txt
  S#a_R_stream_#.txt
  S#a_S_stream_#.txt
  S#a_W_stream_#.txt
  S#b_C_stream_#.txt
  S#b_E_stream_#.tx

## 4. Inventory

In [4]:
from sharp_har.inventory import build_inventory, resolve_duplicate_streams

inventory_df = build_inventory(stage_dir, out_dir=REPO_DIR / "reports")

# Some files are staged from both doppler_traces.zip and
# doppler_traces_S4_S5.zip under the same (trace_id, antenna) — confirmed
# day 1 as two distinct physical recordings (different n_frame), not
# duplicate copies. Split them into distinct traces instead of guessing
# which one to discard. See DUPLICATE_TRACE_SUFFIX in inventory.py.
inventory_df = resolve_duplicate_streams(inventory_df, out_dir=REPO_DIR / "reports")

print("AR-set coverage:", sorted(inventory_df["ar_set"].unique()))
print("dtype distribution:")
print(inventory_df["dtype"].value_counts())
nan_pct = 100 * inventory_df["has_nan"].mean()
print(f"files with NaN: {nan_pct:.1f}%")

2026-07-16 09:00:58,910 [INFO] sharp_har.inventory: inventory.csv written: 408 rows (/content/sharp-har/reports/inventory.csv)
2026-07-16 09:00:58,967 [INFO] sharp_har.inventory: split 8 duplicated (trace_id, antenna) pair(s) into distinct traces: ['S4a_Lalt', 'S5a_Lalt']


AR-set coverage: ['AR-1', 'AR-2', 'AR-3', 'AR-4', 'AR-5', 'AR-6', 'AR-7']
dtype distribution:
dtype
float64    408
Name: count, dtype: int64
files with NaN: 0.0%


## 5. Day-1 checks / gate

Axes, AR-set coverage, activity×AR-set contingency, NaN policy
(stop if >5%), class decision, window-count sanity check.

In [5]:
from sharp_har.inventory import (
    assert_axes, assert_no_duplicate_files, assert_coverage, build_contingency_table,
    apply_nan_policy, exclude_non_activities, decide_classes,
)
from sharp_har.windowing import (
    count_windows, EXPECTED_WINDOWS_TRAIN_STRIDE, EXPECTED_WINDOWS_EVAL_STRIDE,
    TRAIN_STRIDE, EVAL_STRIDE,
)

assert_axes(inventory_df)
assert_no_duplicate_files(inventory_df)  # safety net after resolve_duplicate_streams

missing = assert_coverage(inventory_df)
assert not missing, f"missing AR-sets: {missing} — blocker, resolve before freezing the splits"

# Contingency table on the raw inventory (audit trail: shows anomalies
# like non-activity codes before they get excluded below).
contingency = build_contingency_table(inventory_df, REPO_DIR / "reports" / "contingency.csv")
print(contingency)

inventory_clean = apply_nan_policy(inventory_df, out_dir=REPO_DIR / "reports")

# Drop calibration/reference recordings that matched the naming regex but
# are not HAR activities (e.g. "LOS") — confirmed day-1 by inspecting the
# raw signal (near-static, unlike any real activity). See NON_ACTIVITY_LABELS.
inventory_clean = exclude_non_activities(inventory_clean, out_dir=REPO_DIR / "reports")

classes_info = decide_classes(inventory_clean)
print(classes_info)

sample_n_frame = int(inventory_clean["n_frame"].median())
print("expected windows (train stride, median n_frame):",
      count_windows(sample_n_frame, stride=TRAIN_STRIDE), "expected ~", EXPECTED_WINDOWS_TRAIN_STRIDE)
print("expected windows (eval stride, median n_frame):",
      count_windows(sample_n_frame, stride=EVAL_STRIDE), "expected ~", EXPECTED_WINDOWS_EVAL_STRIDE)

2026-07-16 09:01:15,470 [INFO] sharp_har.inventory: traces excluded for NaN: 0/102 (0.0%)
2026-07-16 09:01:15,486 [INFO] sharp_har.inventory: excluded 1 non-activity trace(s) ['LOS']: ['S5a_LOS']


attivita  C  E  H  J  L  LOS  R  S  W
ar_set                               
AR-1      3  3  3  6  3    0  3  3  3
AR-2      0  2  0  3  2    0  2  0  2
AR-3      0  1  1  2  1    0  1  1  1
AR-4      2  2  2  4  3    0  2  1  2
AR-5      2  1  2  2  2    1  1  0  1
AR-6      1  2  1  4  2    0  2  1  2
AR-7      1  1  1  3  2    0  1  1  1
{'n_att': 8, 'labels': ['C', 'E', 'H', 'J', 'L', 'R', 'S', 'W'], 'c0_paper_set': ['walking', 'running', 'jumping', 'sitting', 'empty']}
expected windows (train stride, median n_frame): 186 expected ~ 197
expected windows (eval stride, median n_frame): 55 expected ~ 58


## 6. P2-lab split (primary rotation, C1–C4)

Test = AR-7 (laboratory, S7 - the paper's hardest set). Val = 15% of train, stratified. Rare cells
pinned. Writes `splits/p2_lab.json` only if the per-cell
coverage assert passes.

In [6]:
from sharp_har.splits import build_p2_rotation

p2_payload = build_p2_rotation(
    inventory_clean,
    out_path=REPO_DIR / "splits" / "p2_lab.json",
    labels=classes_info["labels"],
)
print("train:", len(p2_payload["train"]), "val:", len(p2_payload["val"]), "test:", len(p2_payload["test"]))
print("norm:", p2_payload["norm"])

2026-07-16 09:01:32,953 [INFO] sharp_har.splits: /content/sharp-har/splits/p2_lab.json written (P2-lab): train=81 val=9 test=11 pinned=40


train: 81 val: 9 test: 11
norm: {'mu': 0.11829386682494482, 'sigma': 0.1801231251153305}


## 7. P1-SHARP split (reproduction, C0)

Train = bedroom S1 (AR-1), all campaigns a/b/c. Val = 20% of S1. Test = the SHARP
generalization set. Writes `splits/p1_sharp.json`.

In [7]:
from sharp_har.splits import build_p1_sharp

p1_payload = build_p1_sharp(
    inventory_clean,
    out_path=REPO_DIR / "splits" / "p1_sharp.json",
    c0_paper_set=classes_info["c0_paper_set"],
)
print("train:", len(p1_payload["train"]), "val:", len(p1_payload["val"]), "test:", len(p1_payload["test"]))
print("norm:", p1_payload["norm"])

2026-07-16 09:01:40,263 [INFO] sharp_har.splits: p1_sharp.json written: train=22 val=5 test=74


train: 22 val: 5 test: 74
norm: {'mu': 0.1177432592435071, 'sigma': 0.18022860661778653}


## 8. Final commit

The artifacts produced in this run are **frozen** once
committed (§0.1 — never touched again):

- `splits/p2_lab.json`, `splits/p1_sharp.json`
- `reports/inventory.csv`, `reports/contingency.csv`,
  `reports/excluded_traces.csv`, `reports/name_to_arset.json`

```bash
cd sharp-har
git add splits/*.json reports/*.csv reports/*.json
git commit -m "day 1: inventory + frozen splits (p2_lab, p1_sharp)"
git push
```